# SPINE-GPE v7 — PNAD COVID Certification Engine v1.0.1
Certificação da ponte pandêmica sem identificação falsa de plataforma.

In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
from pathlib import Path
import subprocess, sys, json, shutil
ROOT = Path("/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7")
SCRIPT_CANDIDATES = [Path("/content/SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.1.py"), ROOT / "scripts/SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.1.py"]
REQ_CANDIDATES = [Path("/content/requirements_SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.1.txt"), ROOT / "scripts/requirements_SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.1.txt"]
SCRIPT = next((p for p in SCRIPT_CANDIDATES if p.exists()), SCRIPT_CANDIDATES[0])
REQ = next((p for p in REQ_CANDIDATES if p.exists()), REQ_CANDIDATES[0])
print("ROOT:", ROOT)
print("SCRIPT:", SCRIPT, SCRIPT.exists())
print("REQ:", REQ, REQ.exists())
if not SCRIPT.exists(): raise FileNotFoundError(f"Script não encontrado: {SCRIPT_CANDIDATES}")
if not REQ.exists(): raise FileNotFoundError(f"Requirements não encontrado: {REQ_CANDIDATES}")


ROOT: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7
SCRIPT: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.1.py True
REQ: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/requirements_SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.1.txt True


In [3]:
subprocess.run([sys.executable,"-m","pip","install","-q","--prefer-binary","-r",str(REQ)],check=True)
subprocess.run([sys.executable,"-m","py_compile",str(SCRIPT)],check=True)
print("py_compile: OK")


py_compile: OK


## Auditoria — setembro de 2020

In [4]:
audit = subprocess.run([sys.executable,str(SCRIPT),"--root",str(ROOT),"--mode","audit","--months","9","--download","--strict"],text=True,capture_output=True)
print(audit.stdout); print(audit.stderr); print("Exit code:",audit.returncode)


2026-07-21 00:32:09,315 | INFO | SPINE-GPE PNAD COVID Certifier v1.0.1 | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 | mode=audit | months=[9]
2026-07-21 00:32:18,650 | INFO | Auditoria PNAD COVID concluída | status=AUDIT_PASSED


Exit code: 0


## Certificação — setembro de 2020

In [5]:
cert = subprocess.run([sys.executable,str(SCRIPT),"--root",str(ROOT),"--mode","certify","--months","9","--download","--chunk-rows","50000","--strict"],text=True,capture_output=True)
print(cert.stdout); print(cert.stderr); print("Exit code:",cert.returncode)


2026-07-21 00:32:25,649 | INFO | SPINE-GPE PNAD COVID Certifier v1.0.1 | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 | mode=certify | months=[9]
2026-07-21 00:32:30,569 | INFO | Extraindo PNAD_COVID_092020.csv -> /content/spine_pnad_covid_cache/PNAD_COVID_092020.csv
2026-07-21 00:32:37,251 | INFO | Mês 09/2020: 50000 registros processados
2026-07-21 00:32:40,617 | INFO | Mês 09/2020: 100000 registros processados
2026-07-21 00:32:43,552 | INFO | Mês 09/2020: 150000 registros processados
2026-07-21 00:32:46,554 | INFO | Mês 09/2020: 200000 registros processados
2026-07-21 00:32:50,582 | INFO | Mês 09/2020: 250000 registros processados
2026-07-21 00:32:54,835 | INFO | Mês 09/2020: 300000 registros processados
2026-07-21 00:32:57,880 | INFO | Mês 09/2020: 350000 registros processados
2026-07-21 00:33:00,171 | INFO | Mês 09/2020: 387298 registros processados
2026-07-21 00:33:07,430 | INFO | Certificação PNAD COVID concluída | status=CERTIFIED | relatório=/content/drive/MyDriv

In [6]:
LOCK = ROOT / "00_admin/PNAD_COVID_CERTIFICATION_LOCK.json"
if not LOCK.exists():
    raise RuntimeError(
        "O lock ainda não foi criado. Revise o STDOUT/STDERR e o exit code da célula de certificação antes de continuar."
    )
lock = json.loads(LOCK.read_text(encoding="utf-8"))
print(json.dumps(lock, ensure_ascii=False, indent=2))
if lock.get("status") not in {"CERTIFIED", "CORE_CERTIFIED"}:
    raise RuntimeError(
        f"Certificação não liberada: {lock.get('status')}. Falhas: {lock.get('critical_failures', [])}"
    )
print("STATUS:", lock["status"])


{
  "run_id": "20260721T003225Z",
  "script_version": "1.0.1",
  "schema_version": "spine-gpe-v7-pnad-covid-delivery-1.0.0",
  "validation_schema_version": "spine-gpe-v7-pnad-covid-validation-1.0.0",
  "mode": "certify",
  "status": "CERTIFIED",
  "critical_failures": [],
  "secondary_failures": [],
  "warnings": [],
  "report": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/06_reports/pnad_covid_certification/pnad_covid_certification_report.md",
  "output": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/03_processed/20_pnad_covid_certified/certified_pnad_covid_delivery_2020.parquet",
  "manifest": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/registry/certified_pnad_covid_delivery_2020_manifest.json",
  "layout": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/registry/pnad_covid_layout_2020.json",
  "survey_design": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/registry/survey_design_pnad_covid_2020.json",
  "golden_

## Série completa maio–novembro (após validar setembro)

In [7]:
 full = subprocess.run([sys.executable,str(SCRIPT),"--root",str(ROOT),"--mode","certify","--months","all","--download","--chunk-rows","50000","--strict"],check=False)
